# AnchorFlow — Google Colab 실행 노트북

| 섹션 | 내용 |
|---|---|
| 0 | GPU / 환경 확인 |
| 1 | 레포 클론 & 의존성 설치 |
| 2 | Google Drive 마운트 & 경로 설정 |
| 3 | AMT 사전학습 가중치 다운로드 |
| 4 | 학습 데이터 준비 (MIDI → pkl) |
| 5 | 학습 Config 생성 |
| 6 | Phase 1 학습 (임베딩 적응) |
| 7 | Phase 2 학습 (전체 파인튜닝) |
| 8 | 추론 & MIDI 다운로드 |
| 9 | 체크포인트 관리 |

> ⚠️ **시작 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택  
> ⚠️ **순서대로** 실행하세요. 각 섹션은 이전 섹션 변수에 의존합니다.

---
## 0. GPU / 환경 확인

In [ ]:
!nvidia-smi
!python --version

---
## 1. 레포 클론 & 의존성 설치

In [ ]:
REPO_URL = "https://github.com/olavvn/Ambient.git"
BRANCH   = "feature/anchorflow-lite"

import os
if os.path.exists("/content/Ambient"):
    %cd /content/Ambient
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/Ambient
    %cd /content/Ambient

!git log --oneline -3

In [ ]:
!pip install -q \
    "transformers>=4.40.0" \
    "huggingface_hub>=0.20.0" \
    "accelerate>=0.27.0" \
    "pretty_midi>=0.2.10" \
    "miditok>=3.0.0" \
    "mido>=1.3.0"

# 설치 확인
!python -c "from src.tokenizer import TSDTokenizer; t=TSDTokenizer(); print('TSDTokenizer OK, vocab_size=', t.vocab_size)"

---
## 2. Google Drive 마운트 & 경로 설정

Drive에 저장하면 런타임이 끊겨도 체크포인트·데이터가 유지됩니다.  
Drive를 쓰지 않으려면 `USE_DRIVE = False` 로 변경하세요.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/AnchorFlow"
else:
    BASE = "/content/AnchorFlow"

import os
HF_CACHE_DIR   = f"{BASE}/hf_cache"
DATA_DIR       = f"{BASE}/data/processed"
CKPT_DIR       = f"{BASE}/checkpoints"
MIDI_DIR       = f"{BASE}/midi_input"
OUTPUT_DIR     = f"{BASE}/outputs"
COLAB_CFG_PATH = "/content/Ambient/configs/colab_training_config.yaml"

os.makedirs(f"{DATA_DIR}/train", exist_ok=True)
os.makedirs(f"{DATA_DIR}/val",   exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(MIDI_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

# !python 서브프로세스에서도 HF 캐시 경로를 인식하도록 환경변수 고정
%env HF_HOME={HF_CACHE_DIR}
%env TRANSFORMERS_CACHE={HF_CACHE_DIR}

print("경로 설정 완료")
print(f"  데이터:     {DATA_DIR}")
print(f"  체크포인트: {CKPT_DIR}")
print(f"  HF 캐시:    {HF_CACHE_DIR}")

---
## 3. AMT 사전학습 가중치 다운로드

`stanford-crfm/music-medium-800k` (약 1.4 GB) 를 HuggingFace Hub에서 다운로드합니다.  
Drive 캐시에 이미 있으면 다음 세션부터 수 초 안에 완료됩니다.

In [ ]:
!python scripts/download_pretrained.py \
    --model_id stanford-crfm/music-medium-800k \
    --cache_dir {HF_CACHE_DIR}

---
## 4. 학습 데이터 준비 (MIDI → pkl)

앰비언트 MIDI 파일을 TSDTokenizer로 토큰화해 `pkl` 로 변환합니다.

- **옵션 A** — 로컬 MIDI 업로드 후 변환 ← 아래 두 셀 실행
- **옵션 B** — Drive에 이미 MIDI가 있으면 `MIDI_DIR` 경로만 맞추고 변환 셀 실행
- **옵션 C** — pkl이 이미 준비된 경우 `DATA_DIR/train`, `DATA_DIR/val` 에 복사 후 이 섹션 건너뜀

In [ ]:
# ── 옵션 A: 로컬 MIDI 파일 업로드 ────────────────────────────────────────
from google.colab import files as colab_files
import os

print("MIDI 파일을 선택하세요 (여러 파일 동시 선택 가능, .mid / .midi)")
uploaded = colab_files.upload()

for fname, data in uploaded.items():
    dest = os.path.join(MIDI_DIR, fname)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"  저장: {dest}")

!ls -lh {MIDI_DIR}

In [ ]:
# ── MIDI → pkl 변환 ───────────────────────────────────────────────────────
!python scripts/preprocess_data.py \
    --midi_dir    {MIDI_DIR} \
    --out_dir     {DATA_DIR} \
    --train_ratio 0.9

In [ ]:
# ── 데이터 확인 ───────────────────────────────────────────────────────────
!echo "=== train ==" && ls {DATA_DIR}/train | wc -l
!echo "=== val ===" && ls {DATA_DIR}/val   | wc -l

!python -c "
import pickle, pathlib
train_dir = '{DATA_DIR}/train'
pkls = sorted(pathlib.Path(train_dir).glob('*.pkl'))
if pkls:
    d = pickle.load(open(pkls[0], 'rb'))
    print(f'샘플 파일: {pkls[0].name}')
    print(f'토큰 수:   {len(d[\"tokens\"])}')
    print(f'첫 토큰:   {d[\"tokens\"][:8]}')
else:
    print('pkl 파일 없음 — 변환 셀을 다시 실행하세요')
"

---
## 5. 학습 Config 생성

Colab 환경(경로·배치 크기)에 맞춘 yaml을 생성합니다.  
이 파일이 Phase 1/2 학습 스크립트에 전달됩니다.

In [ ]:
import yaml

colab_cfg = {
    "pretrained": {
        "base_model":        "stanford-crfm/music-medium-800k",
        "resize_embeddings": True,
        "reinit_embeddings": True,
    },
    "data": {
        "train_dir":   f"{DATA_DIR}/train",
        "val_dir":     f"{DATA_DIR}/val",
        "num_workers": 2,
    },
    "phase1": {
        "description":    "임베딩 적응",
        "epochs":          5,
        "lr":              1e-3,
        "freeze_backbone": True,
        "batch_size":      8,     # OOM 발생 시 4로 낮추세요
        "warmup_steps":    200,
    },
    "phase2": {
        "description":           "전체 파인튜닝",
        "epochs":                 25,
        "lr_embedding":           1e-4,
        "lr_backbone":            1e-5,
        "freeze_backbone":        False,
        "batch_size":             4,    # 전체 학습은 메모리 더 사용
        "gradient_accumulation":  4,
        "warmup_steps":           500,
    },
    "common": {
        "optimizer":       "AdamW",
        "weight_decay":    0.01,
        "betas":           [0.9, 0.95],
        "lr_schedule":     "cosine",
        "gradient_clip":   1.0,
        "label_smoothing": 0.1,
        "seq_len":         1024,  # T4 16GB 기준. 여유 있으면 2048
    },
    "output": {
        "checkpoint_dir": CKPT_DIR,
        "save_every":     5,
    },
}

with open(COLAB_CFG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_cfg, f, allow_unicode=True, default_flow_style=False)

print(f"Config 저장: {COLAB_CFG_PATH}")
!cat {COLAB_CFG_PATH}

---
## 6. Phase 1 — 임베딩 적응 학습

트랜스포머 블록을 동결하고 **토큰 임베딩 + LM head만** 학습합니다.  
새 TSD Ambient 어휘를 AMT 블록의 표현 공간에 정렬하는 단계입니다.

In [ ]:
!python training/train_phase1.py --config {COLAB_CFG_PATH}

---
## 7. Phase 2 — 전체 파인튜닝

임베딩/LM head와 트랜스포머 블록 모두 학습합니다.  
임베딩은 높은 LR(`1e-4`), 블록은 낮은 LR(`1e-5`) — ULMFiT 원칙.

In [ ]:
!python training/train_phase2.py \
    --config      {COLAB_CFG_PATH} \
    --phase1_ckpt {CKPT_DIR}/phase1_best.pt

---
## 8. 추론 & MIDI 다운로드

학습된 모델에 anchor MIDI를 넣어 앰비언트를 생성합니다.

In [ ]:
# ── 8-A. Anchor MIDI 업로드 ──────────────────────────────────────────────
from google.colab import files as colab_files
import os

print("Anchor MIDI 파일을 업로드하세요 (.mid)")
uploaded = colab_files.upload()

ANCHOR_PATH = list(uploaded.keys())[0]
with open(ANCHOR_PATH, "wb") as f:
    f.write(uploaded[ANCHOR_PATH])
print(f"Anchor: {ANCHOR_PATH}")

In [ ]:
# ── 8-B. 생성 실행 ───────────────────────────────────────────────────────
INFER_CKPT  = f"{CKPT_DIR}/phase2_best.pt"  # 없으면 phase1_best.pt 로 변경
OUT_MIDI    = f"{OUTPUT_DIR}/generated.mid"
N_TOKENS    = 256   # 생성 토큰 수 (많을수록 길이 증가)
TEMPERATURE = 1.0
TOP_P       = 0.9

!python scripts/inference.py \
    --ckpt        {INFER_CKPT} \
    --anchor      {ANCHOR_PATH} \
    --out         {OUT_MIDI} \
    --n_tokens    {N_TOKENS} \
    --temperature {TEMPERATURE} \
    --top_p       {TOP_P}

In [ ]:
# ── 8-C. 생성된 MIDI 다운로드 ────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(OUT_MIDI)
print(f"다운로드: {OUT_MIDI}")

In [ ]:
# ── 8-D. 더미 Anchor로 빠른 테스트 (anchor MIDI 없을 때) ──────────────────
# C장조 화음(C-E-G)을 더미 anchor로 만들어 추론합니다.
import pretty_midi

dummy = pretty_midi.PrettyMIDI(initial_tempo=60.0)
inst  = pretty_midi.Instrument(program=0)
for pitch in [60, 64, 67]:  # C-E-G
    inst.notes.append(pretty_midi.Note(velocity=64, pitch=pitch, start=0.0, end=2.0))
dummy.instruments.append(inst)

DUMMY_ANCHOR = "/content/dummy_anchor.mid"
dummy.write(DUMMY_ANCHOR)
print(f"더미 anchor 생성: {DUMMY_ANCHOR}")

DUMMY_OUT = f"{OUTPUT_DIR}/dummy_generated.mid"
INFER_CKPT = f"{CKPT_DIR}/phase2_best.pt"

!python scripts/inference.py \
    --ckpt     {INFER_CKPT} \
    --anchor   {DUMMY_ANCHOR} \
    --out      {DUMMY_OUT} \
    --n_tokens 128

---
## 9. 체크포인트 관리

In [ ]:
# 체크포인트 목록
!ls -lh {CKPT_DIR}

In [ ]:
# 특정 체크포인트 다운로드
from google.colab import files as colab_files
colab_files.download(f"{CKPT_DIR}/phase2_best.pt")

In [ ]:
# GPU 메모리 현황
!nvidia-smi --query-gpu=name,memory.used,memory.free,memory.total --format=csv